# Pre-JcvPCA Segment / Joint Review

**Workflow**
1. Set folders and frame window below (joint list loads automatically).
2. **Check the joints** you want to review — mapping and QC tables only show markers linked to those joints.
3. Click **Run mapping table** (markers scoped to checked joints).
4. Click **Run full review** then **Show review tables** (CSV decision tables only).
5. Click **Export window data** for JcvPCA-ready parquet/CSV exports (separate from review tables).

**Note:** Full session inventory = 54 bone markers + ~8 labeled segment-pair regions from QC events. After joint selection you see only the subset whose candidate links overlap your checked joints (finger markers appear only if finger links are selected).

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "pre_jvcpca_review").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, display

from pre_jvcpca_review.discovery import resolve_layer2
from pre_jvcpca_review.export_window import (
    MATRIX_IDENTITY_COLUMNS,
    PRIMARY_ROTVEC_COLUMNS,
    export_window_for_jvcpca,
)
from pre_jvcpca_review.load_layer2 import load_link_manifest
from pre_jvcpca_review.notebook_ui import (
    JointSelector,
    display_summary_card,
    display_table,
)

WIDE = widgets.Layout(width="98%")
BTN = widgets.Layout(width="220px", margin="4px")

# --- Inputs ---
LAYER1_DIR = widgets.Text(value=str(PROJECT_ROOT / "reevluate_project"), description="Layer 1:", layout=WIDE)
LAYER2_DIR = widgets.Text(value=str(PROJECT_ROOT / "reevluate_project"), description="Layer 2:", layout=WIDE)
DATADESCRIPTIONS_PATH = widgets.Text(
    value=str(PROJECT_ROOT / "reevluate_project" / "671_T1_P1_R1_Take 2026-01-06 03.57.12 PM_001_DataDescriptions.csv"),
    description="DataDesc:",
    layout=WIDE,
)
OUTPUT_DIR = widgets.Text(
    value=str(PROJECT_ROOT / "outputs" / "pre_jvcpca_review" / "session_window"),
    description="Output:",
    layout=WIDE,
)
FRAME_START = widgets.IntText(value=16000, description="Start frame:", layout=widgets.Layout(width="200px"))
FRAME_END = widgets.IntText(value=17000, description="End frame:", layout=widgets.Layout(width="200px"))

QC_GAP_0P5 = widgets.Checkbox(value=True, description="gap ≥ 0.5 s")
QC_GAP_0P2 = widgets.Checkbox(value=True, description="gap ≥ 0.2 s")
QC_ARTIFACT = widgets.Checkbox(value=True, description="artifact_sigma")
QC_SWAP = widgets.Checkbox(value=True, description="segment_swap")
ALLOW_NAN_MATRIX = widgets.Checkbox(
    value=False,
    description="Allow NaN in JcvPCA matrix (do not impute)",
    tooltip=(
        "Unchecked (default): export fails if any filtered-analysis rotvec is NaN. "
        "Checked: export proceeds and keeps NaNs in the matrix; counts are recorded in the manifest. "
        "Does not fill or replace missing values."
    ),
)

status_html = widgets.HTML(value="<i>Ready — joint list loads when you open this cell.</i>")
joint_selector: JointSelector | None = None
out_mapping = widgets.Output()
out_review = widgets.Output()
out_export = widgets.Output()
out_tables = widgets.Output()


def _selected_qc_types() -> list[str]:
    types = []
    if QC_GAP_0P5.value:
        types.append("gap_0p5")
    if QC_GAP_0P2.value:
        types.append("gap_0p2")
    if QC_ARTIFACT.value:
        types.append("artifact_sigma")
    if QC_SWAP.value:
        types.append("segment_swap")
    return types


def _dd_arg() -> list[str]:
    path = Path(DATADESCRIPTIONS_PATH.value)
    return ["--datadescriptions", str(path)] if path.is_file() else []


def _run_backend(args: list[str]) -> None:
    cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "build_pre_jvcpca_review.py"), *args]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)


def reload_joint_list(_=None) -> None:
    global joint_selector
    try:
        links = load_link_manifest(resolve_layer2(Path(LAYER2_DIR.value)).link_manifest)
    except Exception as exc:
        status_html.value = f"<span style='color:red'>Could not load joints: {exc}</span>"
        return
    joint_selector = JointSelector.from_links(links)
    n_core = sum(1 for link in links if link.feature_scope == "core_candidate")
    status_html.value = (
        f"<b>{len(links)}</b> joints loaded from Layer 2 folder "
        f"(<b>{n_core}</b> core_candidate). Check joints below, then use the action buttons."
    )
    joint_panel.children = (
        widgets.HTML("<b>Select joints for review</b> (checkbox = include in review tables)"),
        joint_selector.filter_core_only,
        widgets.HBox([btn_select_core, btn_clear_joints]),
        joint_selector.container,
    )


def on_select_core(_btn) -> None:
    if joint_selector:
        joint_selector.select_core_candidates()


def on_clear_joints(_btn) -> None:
    if joint_selector:
        joint_selector.clear_selection()


def on_run_mapping(_btn) -> None:
    out_mapping.clear_output(wait=True)
    if not joint_selector:
        with out_mapping:
            display(HTML("<p style='color:red'>Load joints first (Reload joint list).</p>"))
        return
    selected = joint_selector.selected_link_ids()
    if not selected:
        with out_mapping:
            display(HTML("<p style='color:red'>Select at least one joint checkbox first.</p>"))
        return
    out = Path(OUTPUT_DIR.value)
    out.mkdir(parents=True, exist_ok=True)
    with out_mapping:
        try:
            _run_backend([
                "--layer1-dir", LAYER1_DIR.value,
                "--layer2-dir", LAYER2_DIR.value,
                "--out", str(out),
                "--mapping-only",
                "--selected-links", ",".join(selected),
                *_dd_arg(),
            ])
            df = pd.read_csv(out / "mapping_logic_table.csv")
            display(HTML(
                f"<p><b>{len(df)} markers/regions</b> mapped to candidate links for "
                f"selected joints: {', '.join(selected)}</p>"
            ))
            display_table(df, "mapping_logic_table.csv (joint-scoped)")
        except Exception as exc:
            display(HTML(f"<p style='color:red'>{exc}</p>"))


def on_run_review(_btn) -> None:
    out_review.clear_output(wait=True)
    if not joint_selector:
        with out_review:
            display(HTML("<p style='color:red'>Load joints first (Reload joint list).</p>"))
        return
    selected = joint_selector.selected_link_ids()
    if not selected:
        with out_review:
            display(HTML("<p style='color:red'>Select at least one joint checkbox.</p>"))
        return
    qc = _selected_qc_types()
    if not qc:
        with out_review:
            display(HTML("<p style='color:red'>Select at least one QC evidence type.</p>"))
        return
    out = Path(OUTPUT_DIR.value)
    out.mkdir(parents=True, exist_ok=True)
    with out_review:
        try:
            _run_backend([
                "--layer1-dir", LAYER1_DIR.value,
                "--layer2-dir", LAYER2_DIR.value,
                "--out", str(out),
                "--start-frame", str(FRAME_START.value),
                "--end-frame", str(FRAME_END.value),
                "--selected-links", ",".join(selected),
                "--qc-evidence", ",".join(qc),
                *_dd_arg(),
            ])
            display(HTML(
                f"<p><b>Review complete.</b> {len(selected)} joints, frames "
                f"{FRAME_START.value}–{FRAME_END.value}. Output: <code>{out}</code></p>"
                f"<p>Selected: {', '.join(selected)}</p>"
            ))
        except Exception as exc:
            display(HTML(f"<p style='color:red'>{exc}</p>"))


def on_export_window(_btn) -> None:
    out_export.clear_output(wait=True)
    if not joint_selector:
        with out_export:
            display(HTML("<p style='color:red'>Load joints first (Reload joint list).</p>"))
        return
    selected = joint_selector.selected_link_ids()
    if not selected:
        with out_export:
            display(HTML("<p style='color:red'>Select at least one joint checkbox.</p>"))
        return
    out = Path(OUTPUT_DIR.value)
    out.mkdir(parents=True, exist_ok=True)
    with out_export:
        try:
            paths = export_window_for_jvcpca(
                layer1_dir=Path(LAYER1_DIR.value),
                layer2_dir=Path(LAYER2_DIR.value),
                out_dir=out,
                frame_start=int(FRAME_START.value),
                frame_end=int(FRAME_END.value),
                selected_link_ids=selected,
                allow_nan_matrix=ALLOW_NAN_MATRIX.value,
            )
            import json

            manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
            matrix_df = pd.read_parquet(paths["jvcpca_matrix"])
            flag_log = pd.read_csv(paths["flag_log"])
            feature_cols = [c for c in matrix_df.columns if c not in MATRIX_IDENTITY_COLUMNS]
            display(HTML(
                f"<p><b>Export complete.</b> Frames {FRAME_START.value}–{FRAME_END.value}</p>"
                f"<ul>"
                f"<li><code>{paths['long_rotvec'].name}</code></li>"
                f"<li><code>{paths['jvcpca_matrix'].name}</code></li>"
                f"<li><code>{paths['flag_log'].name}</code></li>"
                f"<li><code>{paths['manifest'].name}</code></li>"
                f"</ul>"
                f"<p>Long rotvec rows: <b>{manifest['long_rotvec_row_count']}</b> · "
                f"Matrix shape: <b>{manifest['jvcpca_matrix_row_count']} × "
                f"{manifest['jvcpca_matrix_column_count']}</b> · "
                f"Flag log rows: <b>{manifest['flag_log_row_count']}</b></p>"
                f"<p><b>selected_link_order</b> ({manifest['selected_link_order_source']}): "
                f"{', '.join(manifest['selected_link_order'])}</p>"
                f"<p><b>Primary rotvec columns:</b> {', '.join(PRIMARY_ROTVEC_COLUMNS)}</p>"
                f"<p style='font-size:12px;color:#555;'>Layer 1 <code>l1_frame_*</code> columns "
                f"repeat per link at the same frame (regional QC, not link-specific proof).</p>"
            ))
            preview_cols = MATRIX_IDENTITY_COLUMNS + feature_cols[: min(6, len(feature_cols))]
            display_table(matrix_df[preview_cols], "window_jvcpca_matrix.parquet (preview)", max_rows=8)
            display_table(flag_log, "window_joint_frame_flag_log.csv (preview)", max_rows=12)
        except Exception as exc:
            display(HTML(f"<p style='color:red'>{exc}</p>"))


def on_show_tables(_btn) -> None:
    out_tables.clear_output(wait=True)
    out = Path(OUTPUT_DIR.value)
    with out_tables:
        summary_path = out / "window_decision_summary.csv"
        if not summary_path.is_file():
            display(HTML("<p style='color:red'>Run full review first.</p>"))
            return
        display_summary_card(pd.read_csv(summary_path), "window_decision_summary.csv")
        for name, max_rows in [
            ("qc_evidence_summary_table.csv", None),
            ("link_joint_review_table.csv", None),
            ("qc_event_review_table.csv", 200),
        ]:
            path = out / name
            if path.is_file():
                display_table(pd.read_csv(path), name, max_rows=max_rows)


btn_reload = widgets.Button(description="Reload joint list", button_style="info", layout=BTN)
btn_select_core = widgets.Button(description="Select core joints", layout=BTN)
btn_clear_joints = widgets.Button(description="Clear joint selection", layout=BTN)
btn_mapping = widgets.Button(description="Run mapping table", button_style="primary", layout=BTN)
btn_review = widgets.Button(description="Run full review", button_style="success", layout=BTN)
btn_export = widgets.Button(description="Export window data", button_style="info", layout=BTN)
btn_tables = widgets.Button(description="Show review tables", button_style="warning", layout=BTN)

btn_reload.on_click(reload_joint_list)
btn_select_core.on_click(on_select_core)
btn_clear_joints.on_click(on_clear_joints)
btn_mapping.on_click(on_run_mapping)
btn_review.on_click(on_run_review)
btn_export.on_click(on_export_window)
btn_tables.on_click(on_show_tables)

joint_panel = widgets.VBox([])

display(
    widgets.VBox([
        widgets.HTML("<h2>Inputs</h2>"),
        LAYER1_DIR,
        LAYER2_DIR,
        DATADESCRIPTIONS_PATH,
        OUTPUT_DIR,
        widgets.HBox([FRAME_START, FRAME_END]),
        widgets.HTML("<b>QC evidence types</b>"),
        widgets.HBox([QC_GAP_0P5, QC_GAP_0P2, QC_ARTIFACT, QC_SWAP]),
        widgets.HTML("<b>Window export options</b>"),
        ALLOW_NAN_MATRIX,
        widgets.HTML(
            "<p style='font-size:12px;color:#555;margin-top:0;'>"
            "<b>Export window data</b> writes four files for the selected frame window and checked joints: "
            "long rotvec parquet, wide JcvPCA matrix parquet, QC flag log CSV, and manifest JSON. "
            "It does not run JcvPCA and does not center/scale the matrix."
            "</p>"
        ),
        status_html,
        btn_reload,
        joint_panel,
        widgets.HTML("<h2>Actions</h2>"),
        widgets.HBox([btn_mapping, btn_review, btn_export, btn_tables]),
        widgets.HTML(
            "<ol style='font-size:12px;color:#444;'>"
            "<li><b>Run mapping table</b> — marker topology (before/after joint pick)</li>"
            "<li><b>Run full review</b> — build review CSVs for checked joints + frame window</li>"
            "<li><b>Export window data</b> — long rotvec parquet, JcvPCA matrix, flag log CSV, manifest JSON</li>"
            "<li><b>Show review tables</b> — display CSV review results below</li>"
            "</ol>"
        ),
        widgets.HTML("<h2>Mapping output</h2>"),
        out_mapping,
        widgets.HTML("<h2>Review status</h2>"),
        out_review,
        widgets.HTML("<h2>Window export</h2>"),
        out_export,
        widgets.HTML("<h2>Review tables</h2>"),
        out_tables,
    ])
)

reload_joint_list()